In [9]:
import torch

def surprisal(probability):
    """확률이 주어진 사건 하나의 놀라움을 계산한다"""
    return - torch.log(
        torch.as_tensor(probability) # -logP(j)
    )
    
    
def entropy(probabilities):
    """확률분포 자체의 평균 놀라움을 계산"""
    terms = torch.where(
        probabilities > 0,
        - probabilities * probabilities.log(), # -P(j) * logP(j)
        torch.zeros_like(probabilities),
    )
    
    return terms.sum()
    
    
def cross_entropy(true_distribution, predicted_distribution):
    """실제 분포와 예측 분포 사이의 교차 엔트로피를 계산한다."""
    # -P(j) * logQ(j)
    return -(
        true_distribution 
        * predicted_distribution.log()
    ).sum()
    

In [10]:
# 발생할 가능성이 낮다고 생각한 사건일수록
# 실제로 발생했을 때 놀라움이 크다.

probabilities = torch.tensor([
    1.0,
    0.5,
    0.1,
    0.01,
])

surprisals = surprisal(probabilities)

for probability, value in zip(
    probabilities,
    surprisals,
):
    print(
        f"Probability={probability.item():.2f}, "
        f"Surprisal={value.item():.4f} nats"
    )

Probability=1.00, Surprisal=-0.0000 nats
Probability=0.50, Surprisal=0.6931 nats
Probability=0.10, Surprisal=2.3026 nats
Probability=0.01, Surprisal=4.6052 nats


In [11]:
# 결과가 확실한 분포
certain_distribution = torch.tensor([
    0.05,
    0.95,
])

# 결과가 불확실한 균등분포
uniform_distribution = torch.tensor([
    0.5,
    0.5,
])

certain_entropy = entropy(
    certain_distribution
)

uniform_entropy = entropy(
    uniform_distribution
)

print("Certain distribution entropy:")
print(certain_entropy)

print("\nUniform distribution entropy:")
print(uniform_entropy)

assert certain_entropy < uniform_entropy

Certain distribution entropy:
tensor(0.1985)

Uniform distribution entropy:
tensor(0.6931)


In [ ]:
# 실제 데이터가 생성되는 확률분포
true_distribution = torch.tensor([
    0.7,
    0.2,
    0.1,
])

# 실제 분포를 정확히 예측한 모델
good_prediction = torch.tensor([
    0.7,
    0.2,
    0.1,
])

# 모든 클래스를 같은 확률로 예측한 모델
uncertain_prediction = torch.tensor([
    1 / 3,
    1 / 3,
    1 / 3,
])

true_entropy = entropy(
    true_distribution
)

good_cross_entropy = cross_entropy(
    true_distribution,
    good_prediction,
)

# 실제 데이터는 P에서 발생하지만, 모델은 Q라고 믿을 때 경험하는 엔트로피
uncertain_cross_entropy = cross_entropy(
    true_distribution,
    uncertain_prediction,
)

print("Entropy H(P):")
print(true_entropy)

print("\nCross-entropy H(P, good Q):")
print(good_cross_entropy)

print("\nCross-entropy H(P, uncertain Q):")
print(uncertain_cross_entropy)

assert torch.allclose(
    true_entropy,
    good_cross_entropy,
)

assert good_cross_entropy < uncertain_cross_entropy

Entropy H(P):
tensor(0.8018)

Cross-entropy H(P, good Q):
tensor(0.8018)

Cross-entropy H(P, uncertain Q):
tensor(1.0986)
